# Colebrook SI-DL example

This notebook reproduces the complete 10,000-point Colebrook experiment. It generates the physical data, performs the one- and two-input SI-DL searches, evaluates the reference dimensionless groups, scans the two-angle Sobol landscape, and writes the final figures.

All CSV files in `csv/` are outputs of the calculations below. The notebook does not require any precomputed Colebrook result file.

The full default run is intentionally substantial: the three-seed differential-evolution searches and both theta scans can take roughly one to two hours. The `pi` conda environment is used throughout.

## 1. Imports and experiment settings

The default settings reproduce the retained 10,000-point result. Environment variables may be used for a small smoke test, but a normal notebook run uses the full values shown here.

In [ ]:
from __future__ import annotations

import os
import sys
import textwrap
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize
from scipy.interpolate import griddata
from scipy.optimize import differential_evolution

WORK_DIR = Path.cwd()
if not (WORK_DIR / "colebrook.ipynb").exists() and (WORK_DIR / "Colebrook" / "colebrook.ipynb").exists():
    WORK_DIR = WORK_DIR / "Colebrook"

CSV_DIR = WORK_DIR / "csv"
FIG_DIR = WORK_DIR / "figures"
CSV_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SI_DIR = WORK_DIR.parent / "SI-DL-main"
if str(SI_DIR) not in sys.path:
    sys.path.insert(0, str(SI_DIR))
import SI_DL

N_SAMPLES = int(os.environ.get("COLEBROOK_N_SAMPLES", "10000"))
RANDOM_STATE = 42
SI_BANDWIDTH = 0.06
DE_BOUNDS = (-4.0, 4.0)
DE_MAXITER = int(os.environ.get("COLEBROOK_DE_MAXITER", "80"))
DE_POPSIZE = int(os.environ.get("COLEBROOK_DE_POPSIZE", "20"))
DE_TOL = float(os.environ.get("COLEBROOK_DE_TOL", "1e-7"))
DE_POLISH = os.environ.get("COLEBROOK_DE_POLISH", "1") != "0"
DE_MUTATION = (0.5, 1.9)
DE_RECOMBINATION = 0.95
DE_SEEDS = [0, 42, 4209]

THETA_GRID_N = int(os.environ.get("COLEBROOK_THETA_GRID_N", "51"))
DIAGONAL_CENTER_N = int(os.environ.get("COLEBROOK_DIAGONAL_CENTER_N", "121"))
DIAGONAL_OFFSET_N = int(os.environ.get("COLEBROOK_DIAGONAL_OFFSET_N", "31"))
DIAGONAL_OFFSET_LIMIT = float(os.environ.get("COLEBROOK_DIAGONAL_OFFSET_LIMIT", "0.075"))

VARIABLE_LABELS = ["u", "rho", "D", "k_s", "mu"]
D_IN = np.matrix("1 -3 1 1 -1; -1 0 0 0 -1; 0 1 0 0 1")

## 2. Generate the Colebrook data

Reynolds number and relative roughness are sampled logarithmically. Diameter is sampled independently, after which velocity and absolute roughness are reconstructed. The Colebrook iteration supplies the friction coefficient $C_f$.

Running this section overwrites `csv/data.csv` with a newly generated, deterministic dataset.

In [ ]:
def colebrook(reynolds_number: np.ndarray, relative_roughness: np.ndarray) -> np.ndarray:
    re = np.asarray(reynolds_number, dtype=float)
    rr = np.asarray(relative_roughness, dtype=float)
    f = np.full_like(re, 0.02, dtype=float)
    for _ in range(100):
        f_new = 1.0 / (-np.log10(rr / 3.7 + 5.02 / (re * np.sqrt(f))))
        if np.max(np.abs(f_new - f)) < 1e-6:
            f = f_new
            break
        f = f_new
    return f


def generate_colebrook_data(n_samples: int) -> pd.DataFrame:
    rng = np.random.default_rng(RANDOM_STATE + 10)
    log_re = rng.uniform(3.0, 5.0, n_samples)
    log_relative_roughness = rng.uniform(-8.0, -0.7, n_samples)
    re = 10.0**log_re
    relative_roughness = 10.0**log_relative_roughness
    rho = np.full(n_samples, 200.0)
    mu = np.full(n_samples, 0.001)
    diameter = rng.uniform(1.0, 10.0, n_samples)
    roughness = relative_roughness * diameter
    velocity = re * mu / (rho * diameter)
    cf = colebrook(re, relative_roughness)
    return pd.DataFrame(
        {
            "u": velocity,
            "rho": rho,
            "D": diameter,
            "k_s": roughness,
            "mu": mu,
            "Re": re,
            "relative_roughness": relative_roughness,
            "Cf": cf,
        }
    )

In [ ]:
generated = generate_colebrook_data(N_SAMPLES)
generated.to_csv(CSV_DIR / "data.csv", index=False)

X = generated[VARIABLE_LABELS].to_numpy(float)
y = generated["Cf"].to_numpy(float)
basis = np.asarray(SI_DL.calc_basis(D_IN, 2), dtype=float)

print(f"Generated {len(generated):,} rows -> {CSV_DIR / 'data.csv'}")
generated.head()

## 3. SI-DL scoring and search utilities

The dimensional null-space basis constrains every candidate to be dimensionless. Candidate exponents are normalized before the logarithmic dimensionless coordinates are evaluated.

The SI-DL objective is the covariance-based explained-variance score using the Gaussian-kernel conditional mean estimator. Known physical groups are included as reference rows in the final summary.

In [ ]:
def normalize_exponents(exponents: np.ndarray) -> np.ndarray:
    row = np.asarray(exponents, dtype=float).reshape(-1)
    scale = float(np.max(np.abs(row)))
    if scale <= 1e-12:
        return row
    out = row / scale
    first = np.flatnonzero(np.abs(out) > 1e-10)
    if first.size and out[first[0]] < 0.0:
        out = -out
    return out


def formula_from_exponents(omegas: np.ndarray, decimals: int = 4) -> str:
    lines = []
    for idx, row in enumerate(np.asarray(omegas, dtype=float), start=1):
        terms = []
        for label, value in zip(VARIABLE_LABELS, row):
            value = float(value)
            if abs(value) < 5e-5:
                continue
            terms.append(f"{label}^{value:.{decimals}f}")
        lines.append(f"pi{idx}: " + " * ".join(terms))
    return "\n".join(lines)


def params_to_omegas(params: np.ndarray, basis: np.ndarray, num_input: int) -> np.ndarray:
    params = np.asarray(params, dtype=float).reshape(num_input, basis.shape[0])
    return np.asarray([normalize_exponents(row @ basis) for row in params])


def log_feature_from_omegas(X: np.ndarray, omegas: np.ndarray) -> np.ndarray:
    log_x = np.log(np.asarray(X, dtype=float))
    omega_array = np.asarray(omegas, dtype=float)
    return np.sum(log_x[:, None, :] * omega_array[None, :, :], axis=2)


def evaluate_feature(feature: np.ndarray, y: np.ndarray) -> dict[str, float | int | str]:
    score = SI_DL.explained_variance_score(
        feature,
        y,
        bandwidth=SI_BANDWIDTH,
        estimator="gaussian_kernel",
        standardize=True,
        leave_one_out=True,
        boundary="mirror",
    )
    m_hat = np.asarray(score["m_hat"], dtype=float).reshape(-1)
    residual = np.asarray(y, dtype=float).reshape(-1) - m_hat
    var_y = float(np.var(y, ddof=0))
    mse = float(np.mean(residual**2))
    score.update(
        {
            "mse": mse,
            "rmse": float(np.sqrt(mse)),
            "nrmse_var": float(np.sqrt(mse / var_y)),
            "corr_Y_mhat": float(np.corrcoef(y, m_hat)[0, 1]),
        }
    )
    return score

In [ ]:
def params_for_known_groups(basis: np.ndarray) -> dict[str, np.ndarray]:
    known_re = np.array([1.0, 1.0, 1.0, 0.0, -1.0])
    known_rr = np.array([0.0, 0.0, -1.0, 1.0, 0.0])
    previous_sidl_1 = np.array([1.0, 1.0, 0.23423725086294048, 0.7657627491370596, -1.0])
    previous_sidl_2 = np.array([0.6037203926408655, 0.6037203926408655, 1.0, -0.3962796073591344, -0.6037203926408655])
    rows = {
        "Known Re": known_re,
        "Known k_s/D": known_rr,
        "Previous SI-DL pi1": previous_sidl_1,
        "Previous SI-DL pi2": previous_sidl_2,
    }
    return {
        name: np.linalg.lstsq(basis.T, omega, rcond=None)[0]
        for name, omega in rows.items()
    }


def make_init_population(num_input: int, basis: np.ndarray, seed: int) -> np.ndarray:
    dim = basis.shape[0] * num_input
    pop_n = max(5, DE_POPSIZE * dim)
    rng = np.random.default_rng(seed + 991 * num_input)
    init = rng.uniform(DE_BOUNDS[0], DE_BOUNDS[1], size=(pop_n, dim))
    anchors = params_for_known_groups(basis)
    one_dim = [
        anchors["Known Re"],
        anchors["Known k_s/D"],
        anchors["Previous SI-DL pi1"],
        anchors["Previous SI-DL pi2"],
    ]
    if num_input == 1:
        rows = [row for row in one_dim]
    else:
        rows = [
            np.r_[anchors["Known Re"], anchors["Known k_s/D"]],
            np.r_[anchors["Known k_s/D"], anchors["Known Re"]],
            np.r_[anchors["Previous SI-DL pi1"], anchors["Previous SI-DL pi2"]],
            np.r_[anchors["Previous SI-DL pi2"], anchors["Previous SI-DL pi1"]],
        ]
        for base_row in list(rows):
            for scale in (0.01, 0.03, 0.06):
                rows.append(base_row + rng.normal(0.0, scale, size=dim))
    for idx, row in enumerate(rows[:pop_n]):
        init[idx] = np.clip(row, DE_BOUNDS[0], DE_BOUNDS[1])
    return init

## 4. Differential-evolution searches

Both one-input and two-input models are optimized with seeds 0, 42, and 4209. The best result for each input dimension is retained. This is the time-consuming part of the notebook.

The cell prints one progress line per differential-evolution iteration, so a long run remains observable.

In [ ]:
def run_sidl_de(X: np.ndarray, y: np.ndarray, num_input: int, seed: int) -> dict:
    basis = np.asarray(SI_DL.calc_basis(D_IN, 2), dtype=float)
    bounds = [DE_BOUNDS] * (basis.shape[0] * num_input)
    init = make_init_population(num_input, basis, seed)
    eval_count = 0
    best_score = -np.inf
    best_params = None
    log_rows = []
    started = time.time()

    def objective(params: np.ndarray) -> float:
        nonlocal eval_count, best_score, best_params
        eval_count += 1
        try:
            omegas = params_to_omegas(params, basis, num_input)
            if np.linalg.matrix_rank(omegas, tol=1e-8) < num_input:
                return 1e6
            feature = log_feature_from_omegas(X, omegas)
            if not np.all(np.isfinite(feature)):
                return 1e6
            if np.any(np.std(feature, axis=0, ddof=0) <= 1e-12):
                return 1e6
            score = float(evaluate_feature(feature, y)["S_cov"])
        except Exception:
            return 1e6
        if not np.isfinite(score):
            return 1e6
        if score > best_score:
            best_score = score
            best_params = np.asarray(params, dtype=float).copy()
        return -score

    def callback(xk: np.ndarray, convergence: float) -> bool:
        elapsed = time.time() - started
        omegas = params_to_omegas(xk, basis, num_input)
        feature = log_feature_from_omegas(X, omegas)
        score = evaluate_feature(feature, y)
        row = {
            "num_input": num_input,
            "seed": seed,
            "iteration": len(log_rows) + 1,
            "evaluations": eval_count,
            "elapsed_seconds": elapsed,
            "convergence": float(convergence),
            "S_cov": float(score["S_cov"]),
            "S_cov_raw": float(score["S_cov_raw"]),
            "sidl_error": float(1.0 - score["S_cov"]),
            "nrmse_var": float(score["nrmse_var"]),
            "corr_Y_mhat": float(score["corr_Y_mhat"]),
            "formula": formula_from_exponents(omegas),
        }
        log_rows.append(row)
        print(
            f"k={num_input} seed={seed} iter={row['iteration']:03d} "
            f"eval={eval_count} S_cov={row['S_cov']:.8f} "
            f"err={row['sidl_error']:.3e} elapsed={elapsed:.1f}s",
            flush=True,
        )
        return False

    print(
        f"Starting SI-DL DE: k={num_input}, seed={seed}, n={X.shape[0]}, "
        f"maxiter={DE_MAXITER}, popsize={DE_POPSIZE}, bounds={DE_BOUNDS}, bandwidth={SI_BANDWIDTH}",
        flush=True,
    )
    result = differential_evolution(
        objective,
        bounds=bounds,
        maxiter=DE_MAXITER,
        popsize=DE_POPSIZE,
        tol=DE_TOL,
        mutation=DE_MUTATION,
        recombination=DE_RECOMBINATION,
        seed=seed,
        init=init,
        polish=DE_POLISH,
        updating="immediate",
        workers=1,
        callback=callback,
    )
    params = np.asarray(result.x if result.fun <= -best_score else best_params, dtype=float)
    omegas = params_to_omegas(params, basis, num_input)
    feature = log_feature_from_omegas(X, omegas)
    score = evaluate_feature(feature, y)
    return {
        "num_input": num_input,
        "seed": seed,
        "params": params,
        "omegas": omegas,
        "feature": feature,
        "score": score,
        "optimizer_result": result,
        "log": pd.DataFrame(log_rows),
        "elapsed_seconds": time.time() - started,
        "objective_evaluations": eval_count,
    }

In [ ]:
def summary_row(
    method: str,
    num_input: int,
    seed: float,
    omegas: np.ndarray,
    score: dict,
    result,
    objective_evaluations: int,
    elapsed_seconds: float,
) -> dict:
    return {
        "method": method,
        "num_input": int(num_input),
        "seed": seed,
        "n_samples": N_SAMPLES,
        "bandwidth": SI_BANDWIDTH,
        "bounds": str(DE_BOUNDS),
        "maxiter": DE_MAXITER,
        "popsize": DE_POPSIZE,
        "polish": DE_POLISH,
        "S_cov": float(score["S_cov"]),
        "S_cov_raw": float(score["S_cov_raw"]),
        "sidl_error": float(1.0 - score["S_cov"]),
        "mse": float(score["mse"]),
        "rmse": float(score["rmse"]),
        "nrmse_var": float(score["nrmse_var"]),
        "corr_Y_mhat": float(score["corr_Y_mhat"]),
        "n_retained": int(score["n_retained"]),
        "objective_evaluations": int(objective_evaluations),
        "elapsed_seconds": float(elapsed_seconds),
        "optimizer_success": "" if result is None else bool(result.success),
        "optimizer_message": "" if result is None else str(result.message),
        "formula": formula_from_exponents(omegas),
    }


def evaluate_reference_rows(X: np.ndarray, y: np.ndarray, basis: np.ndarray) -> list[dict]:
    anchors = params_for_known_groups(basis)
    references = {
        "Known Re": anchors["Known Re"].reshape(1, -1),
        "Known k_s/D": anchors["Known k_s/D"].reshape(1, -1),
        "Known [Re, k_s/D]": np.vstack([anchors["Known Re"], anchors["Known k_s/D"]]),
        "Previous SI-DL 2D": np.vstack([anchors["Previous SI-DL pi1"], anchors["Previous SI-DL pi2"]]),
    }
    rows = []
    for method, params in references.items():
        omegas = params_to_omegas(params.reshape(-1), basis, params.shape[0])
        feature = log_feature_from_omegas(X, omegas)
        score = evaluate_feature(feature, y)
        rows.append(summary_row(method, params.shape[0], np.nan, omegas, score, None, 0, 0.0))
    return rows


def exponents_rows(method: str, omegas: np.ndarray) -> list[dict]:
    rows = []
    for idx, row in enumerate(np.asarray(omegas, dtype=float), start=1):
        for label, value in zip(VARIABLE_LABELS, row):
            rows.append(
                {
                    "method": method,
                    "pi_group": f"pi{idx}",
                    "variable": label,
                    "normalized_exponent": float(value),
                }
            )
    return rows

In [ ]:
summary_rows = evaluate_reference_rows(X, y, basis)
exponent_rows = []
best_results = {}

for num_input in (1, 2):
    best = None
    for seed in DE_SEEDS:
        result = run_sidl_de(X, y, num_input=num_input, seed=seed)
        method = f"SI-DL {num_input}D DE seed{seed}"
        row = summary_row(
            method,
            num_input,
            seed,
            result["omegas"],
            result["score"],
            result["optimizer_result"],
            result["objective_evaluations"],
            result["elapsed_seconds"],
        )
        summary_rows.append(row)
        if best is None or row["S_cov"] > best["row"]["S_cov"]:
            best = {"row": row, "result": result}

    best_method = f"SI-DL {num_input}D DE best"
    best_row = dict(best["row"])
    best_row["method"] = best_method
    summary_rows.append(best_row)
    exponent_rows.extend(exponents_rows(best_method, best["result"]["omegas"]))
    best_results[num_input] = best["result"]

reference_params = params_for_known_groups(basis)
reference_omegas = {
    "Known Re": params_to_omegas(reference_params["Known Re"], basis, 1),
    "Known k_s/D": params_to_omegas(reference_params["Known k_s/D"], basis, 1),
    "Known [Re, k_s/D]": params_to_omegas(
        np.r_[reference_params["Known Re"], reference_params["Known k_s/D"]], basis, 2
    ),
    "Previous SI-DL 2D": params_to_omegas(
        np.r_[reference_params["Previous SI-DL pi1"], reference_params["Previous SI-DL pi2"]], basis, 2
    ),
}
for method, omegas in reference_omegas.items():
    exponent_rows.extend(exponents_rows(method, omegas))

summary = pd.DataFrame(summary_rows).sort_values(
    ["num_input", "S_cov"], ascending=[True, False]
).reset_index(drop=True)
exponents = pd.DataFrame(exponent_rows)

summary.to_csv(CSV_DIR / "summary.csv", index=False)
exponents.to_csv(CSV_DIR / "exponents.csv", index=False)

print(f"Wrote {CSV_DIR / 'summary.csv'}")
print(f"Wrote {CSV_DIR / 'exponents.csv'}")
summary[["method", "num_input", "seed", "S_cov", "sidl_error", "rmse", "nrmse_var"]]

## 5. Coarse two-angle Sobol landscape

The two physical directions are $\log(Re)$ and $\log(D/k_s)$. Each theta pair forms two rotated candidate coordinates. The default 51-by-51 grid produces `csv/theta_surface.csv`.

In [ ]:
def scov_for_thetas(theta1, theta2, z_re, z_roughness, response):
    eta1 = np.cos(theta1) * z_re + np.sin(theta1) * z_roughness
    eta2 = np.cos(theta2) * z_re + np.sin(theta2) * z_roughness
    score = SI_DL.explained_variance_score(
        np.column_stack([eta1, eta2]),
        response,
        bandwidth=SI_BANDWIDTH,
        estimator="gaussian_kernel",
        standardize=True,
        leave_one_out=True,
        boundary="mirror",
    )
    return float(score["S_cov"])

z_re = np.log(generated["Re"].to_numpy(float))
z_roughness = np.log(1.0 / generated["relative_roughness"].to_numpy(float))

theta_grid = np.linspace(0.0, np.pi, THETA_GRID_N)
coarse_rows = []
coarse_started = time.time()
for row_index, theta2 in enumerate(theta_grid, start=1):
    for theta1 in theta_grid:
        coarse_rows.append({
            "theta1": float(theta1),
            "theta2": float(theta2),
            "theta1_over_pi": float(theta1 / np.pi),
            "theta2_over_pi": float(theta2 / np.pi),
            "S_cov": scov_for_thetas(theta1, theta2, z_re, z_roughness, y),
        })
    print(
        f"theta grid row {row_index:03d}/{THETA_GRID_N}; "
        f"elapsed={time.time() - coarse_started:.1f}s",
        flush=True,
    )

coarse_surface = pd.DataFrame(coarse_rows)
coarse_surface.to_csv(CSV_DIR / "theta_surface.csv", index=False)
print(f"Wrote {CSV_DIR / 'theta_surface.csv'}")

## 6. Refined diagonal-band scan

The highest-score region lies near $\theta_1=\theta_2$. A denser band scan resolves this region while retaining both absolute angles in the output. The default settings produce 3,751 rows in `csv/theta_diagonal.csv`.

In [ ]:
theta1_grid = np.linspace(0.0, 1.0, DIAGONAL_CENTER_N)
delta_grid = np.linspace(-DIAGONAL_OFFSET_LIMIT, DIAGONAL_OFFSET_LIMIT, DIAGONAL_OFFSET_N)
diagonal_rows = []
diagonal_started = time.time()

for offset_index, delta_over_pi in enumerate(delta_grid, start=1):
    for theta1_over_pi in theta1_grid:
        theta2_over_pi = theta1_over_pi + delta_over_pi
        if 0.0 <= theta2_over_pi <= 1.0:
            score = scov_for_thetas(
                theta1_over_pi * np.pi,
                theta2_over_pi * np.pi,
                z_re,
                z_roughness,
                y,
            )
        else:
            score = np.nan
        diagonal_rows.append({
            "theta1_over_pi": float(theta1_over_pi),
            "theta2_over_pi": float(theta2_over_pi),
            "delta_over_pi": float(delta_over_pi),
            "S_cov": float(score) if np.isfinite(score) else np.nan,
        })
    print(
        f"diagonal row {offset_index:03d}/{DIAGONAL_OFFSET_N}; "
        f"elapsed={time.time() - diagonal_started:.1f}s",
        flush=True,
    )

diagonal_surface = pd.DataFrame(diagonal_rows)
diagonal_surface.to_csv(CSV_DIR / "theta_diagonal.csv", index=False)
print(f"Wrote {CSV_DIR / 'theta_diagonal.csv'}")

## 7. Build the top-view figure

The coarse grid and diagonal refinement are combined before interpolation. The red star marks the best two-input SI-DL result; the square marks the known $[Re,D/k_s]$ pair. The right panel shows the generated data in the learned coordinates.

In [ ]:
# Reload the generated CSV files to verify that the figures depend only on notebook outputs.
generated = pd.read_csv(CSV_DIR / "data.csv")
exponents = pd.read_csv(CSV_DIR / "exponents.csv")
summary = pd.read_csv(CSV_DIR / "summary.csv")
coarse_surface = pd.read_csv(CSV_DIR / "theta_surface.csv")
diagonal_surface = pd.read_csv(CSV_DIR / "theta_diagonal.csv")

best_omegas = []
for pi_group in ("pi1", "pi2"):
    subset = exponents[
        exponents["method"].eq("SI-DL 2D DE best")
        & exponents["pi_group"].eq(pi_group)
    ]
    best_omegas.append([
        float(subset.loc[subset["variable"].eq(label), "normalized_exponent"].iloc[0])
        for label in VARIABLE_LABELS
    ])
best_omegas = np.asarray(best_omegas, dtype=float)

physical_inputs = generated[VARIABLE_LABELS].to_numpy(float)
best_coordinates = np.sum(
    np.log(physical_inputs)[:, None, :] * best_omegas[None, :, :], axis=2
)
eta1, eta2 = best_coordinates.T
cf = generated["Cf"].to_numpy(float)

best_theta = np.asarray([
    np.arctan2(-omega[3], omega[0]) % np.pi for omega in best_omegas
])

surface = pd.concat([
    coarse_surface[["theta1_over_pi", "theta2_over_pi", "S_cov"]],
    diagonal_surface[["theta1_over_pi", "theta2_over_pi", "S_cov"]],
], ignore_index=True)
surface = surface[
    surface["S_cov"].notna()
    & surface["theta1_over_pi"].between(0.0, 1.0)
    & surface["theta2_over_pi"].between(0.0, 1.0)
].drop_duplicates(["theta1_over_pi", "theta2_over_pi"], keep="last")

theta1 = surface["theta1_over_pi"].to_numpy(float)
theta2 = surface["theta2_over_pi"].to_numpy(float)
scov = surface["S_cov"].to_numpy(float)
theta1_grid_plot, theta2_grid_plot = np.meshgrid(
    np.linspace(0.0, 1.0, 520), np.linspace(0.0, 1.0, 520)
)
scov_smooth = griddata(
    np.column_stack([theta1, theta2]), scov,
    (theta1_grid_plot, theta2_grid_plot), method="linear"
)

In [ ]:
plt.rcParams.update({
    "font.size": 15, "font.family": "Times New Roman", "mathtext.fontset": "stix",
    "axes.titlesize": 21, "axes.labelsize": 19, "xtick.labelsize": 15,
    "ytick.labelsize": 15, "legend.fontsize": 14,
})
fig = plt.figure(figsize=(14.8, 6.2), dpi=480)
ax2d = fig.add_axes([0.050, 0.105, 0.405, 0.865])
norm = Normalize(vmin=float(np.nanmin(scov)), vmax=float(np.nanmax(scov)))
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("white")
mesh = ax2d.imshow(
    np.ma.masked_invalid(scov_smooth), origin="lower", extent=[0, 1, 0, 1],
    cmap=cmap, norm=norm, interpolation="bicubic", aspect="auto"
)
ax2d.plot([0, 1], [0, 1], color="black", linewidth=1.6, linestyle="--", label=r"$\theta_1=\theta_2$")
ax2d.scatter(best_theta[0] / np.pi, best_theta[1] / np.pi, marker="*", s=180,
             c="#d62728", edgecolors="black", linewidths=0.8, label="SI-DL 2D best", zorder=4)
ax2d.scatter(0.0, 0.5, marker="s", s=80, c="#ffbf00", edgecolors="black",
             linewidths=0.8, label="Known [Re, D/k_s]", zorder=4)
ax2d.set(xlabel=r"$\theta_1/\pi$", ylabel=r"$\theta_2/\pi$", xlim=(-0.03, 1.03), ylim=(-0.03, 1.03))
ax2d.legend(frameon=True, loc="lower right")

ax3d = fig.add_axes([0.535, 0.030, 0.455, 0.950], projection="3d")
ax3d.scatter(eta1, eta2, cf, color="#123f6d", s=10, alpha=0.72, linewidths=0, depthshade=True)
ax3d.set_xlabel(r"$\log(\Pi_1)$", labelpad=9)
ax3d.set_ylabel(r"$\log(\Pi_2)$", labelpad=9)
ax3d.set_zlabel(r"$C_f$", labelpad=9)
ax3d.view_init(elev=26, azim=-52)
ax3d.set_box_aspect((1.15, 1.0, 0.68))
ax3d.grid(True, alpha=0.25)

cbar_ax = fig.add_axes([0.468, 0.105, 0.022, 0.865])
cbar = fig.colorbar(mesh, cax=cbar_ax)
cbar.set_label("Sobol index", fontsize=18)
fig.savefig(FIG_DIR / "topview.png", bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"Wrote {FIG_DIR / 'topview.png'}")

## 8. Build the summary figure

The summary image is rendered from the newly generated `summary.csv`. Formula text is wrapped and the formula column receives extra width so that no result is clipped.

In [ ]:
view_columns = ["method", "num_input", "seed", "S_cov", "sidl_error", "rmse", "nrmse_var", "formula"]
view = summary[view_columns].copy()
view["formula"] = (view["formula"].astype(str)
    .str.replace(" pi2:", "\npi2:", regex=False)
    .map(lambda text: "\n".join(
        textwrap.fill(part, width=72, break_long_words=False) for part in text.split("\n")
    )))
for column in view.columns:
    if pd.api.types.is_float_dtype(view[column]):
        view[column] = view[column].map(lambda value: f"{value:.6g}")

fig, ax = plt.subplots(figsize=(18.5, 2.2 + 0.78 * len(view)), dpi=220)
ax.axis("off")
ax.set_title("Colebrook 10,000-point SI-DL summary", fontsize=18, weight="bold", pad=14)
column_widths = [0.14, 0.06, 0.06, 0.07, 0.07, 0.07, 0.08, 0.45]
table = ax.table(
    cellText=view.values, colLabels=view.columns, cellLoc="center", colLoc="center",
    colWidths=column_widths, bbox=[0.01, 0.02, 0.98, 0.90]
)
table.auto_set_font_size(False)
for (row, _), cell in table.get_celld().items():
    cell.set_edgecolor("#3f3f46")
    cell.set_linewidth(0.75)
    cell.PAD = 0.025
    if row == 0:
        cell.set_facecolor("#1f2937")
        cell.get_text().set_color("white")
        cell.get_text().set_weight("bold")
        cell.get_text().set_fontsize(8.5)
    else:
        cell.set_facecolor("#f8fafc" if row % 2 else "#ffffff")
        cell.get_text().set_fontsize(7.2)
        if cell.get_text().get_text().startswith("pi1:"):
            cell.get_text().set_ha("left")
fig.savefig(FIG_DIR / "summary.png", bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"Wrote {FIG_DIR / 'summary.png'}")

## Outputs

A complete run creates exactly these reproducible result files:

- `csv/data.csv`
- `csv/summary.csv`
- `csv/exponents.csv`
- `csv/theta_surface.csv`
- `csv/theta_diagonal.csv`
- `figures/topview.png`
- `figures/summary.png`